# Phase 1 — Getting the Data

We download more than ten years of daily price history (open, high, low, close, volume) for every company in the S&P 500, and save each one to disk.

**Why this set of companies, and this length.** The S&P 500 is large, heavily traded and well documented, which makes results credible. Ten-plus years deliberately spans calm markets, the COVID crash and the 2022 downturn, so anything we build later is tested across very different conditions — not just one lucky stretch.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import os
import time
from datetime import datetime

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

START_DATE = "2015-01-01"
END_DATE = datetime.today().strftime("%Y-%m-%d")
BATCH_SIZE = 50
SLEEP_BETWEEN_BATCHES = 2
HISTORY_CUTOFF = "2015-06-01"

print(f"Date range: {START_DATE} to {END_DATE}")

In [ ]:
import requests
from io import StringIO

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
tickers = pd.read_html(StringIO(resp.text))[0]["Symbol"].tolist()
tickers = [t.replace(".", "-") for t in tickers]

print(f"Total tickers: {len(tickers)}")

In [ ]:
failed = []

for i in range(0, len(tickers), BATCH_SIZE):
    batch = tickers[i:i + BATCH_SIZE]
    print(f"Batch {i//BATCH_SIZE + 1}/{len(tickers)//BATCH_SIZE + 1}: {batch[0]}..{batch[-1]}")

    try:
        raw = yf.download(batch, start=START_DATE, end=END_DATE, auto_adjust=True, progress=False)
        for ticker in batch:
            try:
                df = raw.xs(ticker, level=1, axis=1).copy() if len(batch) > 1 else raw.copy()
                df = df.dropna(how="all")
                df.index.name = "date"
                df.columns = [c.lower().replace(" ", "_") for c in df.columns]
                if df.empty:
                    failed.append(ticker)
                    continue
                df.to_parquet(f"data/raw/{ticker}.parquet")
            except Exception as e:
                print(f"  SKIP {ticker} -- {e}")
                failed.append(ticker)
    except Exception as e:
        print(f"Batch failed: {e}")
        failed.extend(batch)

    time.sleep(SLEEP_BETWEEN_BATCHES)

print(f"\nDone. Failed: {len(failed)} {failed}")

In [ ]:
valid_tickers = []
removed_tickers = []

for ticker in tickers:
    path = f"data/raw/{ticker}.parquet"
    if not os.path.exists(path):
        removed_tickers.append(ticker)
        continue
    df = pd.read_parquet(path)
    if df.empty or str(df.index.min().date()) > HISTORY_CUTOFF:
        removed_tickers.append(ticker)
    else:
        valid_tickers.append(ticker)

print(f"Valid: {len(valid_tickers)} | Removed: {len(removed_tickers)}")
print(f"Removed: {removed_tickers}")

In [ ]:
adj_close = pd.DataFrame({t: pd.read_parquet(f"data/raw/{t}.parquet")["close"] for t in valid_tickers})
adj_close = adj_close.sort_index().dropna(how="all").ffill(limit=2)
adj_close = adj_close.loc[:, adj_close.isnull().mean() < 0.02]

log_returns = np.log(adj_close / adj_close.shift(1)).dropna(how="all")

adj_close.to_parquet("data/processed/adj_close.parquet")
log_returns.to_parquet("data/processed/log_returns.parquet")

print(f"Final universe: {adj_close.shape[1]} tickers x {adj_close.shape[0]} days")

## Quick look at the downloaded data

We print the size and date range, confirm there are no gaps, and draw a few sanity-check charts.

**How to read the charts:** the three line charts (Apple, Microsoft, JPMorgan) should look like normal long-run price histories — generally drifting upward with visible dips. The bottom histogram shows Apple's daily moves: it should be a tall bell shape centred near zero (most days are small moves) with a few rare large moves out at the edges. Anything strange here — flat lines, giant spikes, obvious gaps — would point to a download problem.

In [ ]:
import matplotlib.pyplot as plt

print(f"adj_close:   {adj_close.shape}")
print(f"log_returns: {log_returns.shape}")
print(f"Range: {adj_close.index.min().date()} to {adj_close.index.max().date()}")
print(f"Missing: adj_close={adj_close.isnull().sum().sum()}, log_returns={log_returns.isnull().sum().sum()}")

fig, axes = plt.subplots(3, 1, figsize=(14, 10))
for ax, t in zip(axes, ["AAPL", "MSFT", "JPM"]):
    adj_close[t].plot(ax=ax, title=f"{t} Adjusted Close")
plt.tight_layout()
plt.show()

adj_close["AAPL"].pct_change().hist(bins=100, figsize=(10, 4))
plt.title("AAPL Daily Returns Distribution")
plt.show()

## Data Metadata

| Field | Value |
|-------|-------|
| **Data Source** | Yahoo Finance via `yfinance` Python library |
| **Universe** | S&P 500 constituents (as of download date) |
| **Date Range** | 2015-01-02 to 2026-04-17 |
| **Frequency** | Daily OHLCV |
| **Adjustment** | `auto_adjust=True` — prices adjusted for splits and dividends |
| **History Filter** | Tickers must have data starting on or before 2015-06-01 |
| **Missing Data** | Forward-filled (limit=2 days), tickers with >2% missing dropped |
| **Retrieval Date** | 2026-04-18 |
| **Final Ticker Count** | 462 (from 503 scraped, 40 removed for insufficient history, 1 dropped for >2% missing) |
| **Trading Days** | 2,839 |

### Known Limitations

- **Survivorship bias**: The ticker list is the *current* S&P 500 composition, not historical. Companies that were removed from the index (due to delisting, acquisition, or poor performance) are not included. This biases results toward survivors.
- **Corporate actions**: Handled by `auto_adjust=True`, but complex events (spin-offs, mergers) may not be perfectly captured.
- **Delisted tickers**: Not included in the dataset.
- **Data source reliability**: Yahoo Finance is free but occasionally has data gaps or errors for individual tickers.